TRANSFORMACION DE DATOS Y REDUCCION DE FILAS A ENCUENTROS COMPETITIVOS

In [86]:
import pandas as pd
df = pd.read_csv('results.csv')
df.tail(20)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49457,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49458,2026-06-24,Morocco,Haiti,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49459,2026-06-25,United States,Turkey,NaN,NaN,FIFA World Cup,Inglewood,United States,False
49460,2026-06-25,Paraguay,Australia,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
49461,2026-06-25,Curaçao,Ivory Coast,NaN,NaN,FIFA World Cup,Philadelphia,United States,True
49462,2026-06-25,Ecuador,Germany,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49463,2026-06-25,Japan,Sweden,NaN,NaN,FIFA World Cup,Arlington,United States,True
49464,2026-06-25,Tunisia,Netherlands,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49465,2026-06-26,Egypt,Iran,NaN,NaN,FIFA World Cup,Seattle,United States,True
49466,2026-06-26,New Zealand,Belgium,NaN,NaN,FIFA World Cup,Vancouver,Canada,True


ACTUALIZACIÓN DE LOS RESULTADOS

In [87]:
# --- RESULTADOS DEL VIERNES 12 DE JUNIO ---

# Canadá 1 - 1 Bosnia
df.loc[(df['date'] == '2026-06-12') & (df['home_team'] == 'Canada'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-12') & (df['away_team'] == 'Bosnia and Herzegovina'), 'away_score'] = 1

# Estados Unidos 4 - 1 Paraguay
df.loc[(df['date'] == '2026-06-12') & (df['home_team'] == 'United States'), 'home_score'] = 4
df.loc[(df['date'] == '2026-06-12') & (df['away_team'] == 'Paraguay'), 'away_score'] = 1

# --- RESULTADOS DEL SÁBADO 13 DE JUNIO (AYER) ---

# Catar 1 - 1 Suiza
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Qatar'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Switzerland'), 'away_score'] = 1

# Brasil 1 - 1 Marruecos
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Brazil'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Morocco'), 'away_score'] = 1

# Haití 0 - 1 Escocia
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Haiti'), 'home_score'] = 0
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Scotland'), 'away_score'] = 1

# Australia 2 - 0 Turquía
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Australia'), 'home_score'] = 2
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Turkey'), 'away_score'] = 0

# Alemania 7 - 1 Curazao
df.loc[(df['date'] == '2026-06-14') & (df['home_team'] == 'Germany'), 'home_score'] = 7
df.loc[(df['date'] == '2026-06-14') & (df['away_team'] == 'Curaçao'), 'away_score'] = 1

# Paises Bajos 2 - 2 Japon
df.loc[(df['date'] == '2026-06-14') & (df['home_team'] == 'Netherlands'), 'home_score'] = 2
df.loc[(df['date'] == '2026-06-14') & (df['away_team'] == 'Japan'), 'away_score'] = 2

# Costa de Marfil 1 - 0 Ecuador
df.loc[(df['date'] == '2026-06-14') & (df['home_team'] == 'Ivory Coast'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-14') & (df['away_team'] == 'Ecuador'), 'away_score'] = 0

# Suecia 5 - 1 Tunez
df.loc[(df['date'] == '2026-06-14') & (df['home_team'] == 'Sweden'), 'home_score'] = 5
df.loc[(df['date'] == '2026-06-14') & (df['away_team'] == 'Tunisia'), 'away_score'] = 1



print("Resultados cargados en la matriz")

Resultados cargados en la matriz


In [88]:
#Convertir la columna date en formato datetime
df["date"]=pd.to_datetime(df["date"])

In [89]:
#Solo queremos conservar los partidos unicamente desde 1990 en adelante
df_actual=df[df["date"].dt.year >= 1990]

In [90]:
#Eliminamos los partidos amistosos, solo se usaran los partidos oficiales
df_oficial=df_actual[df_actual["tournament"]!="Friendly"].copy()


In [91]:
#Creamos un identificador unico para cada partido
df_oficial["match_id"]=range(len(df_oficial))


In [92]:
print(f"Dataset filtrado con éxito. Total de partidos competitivos: {len(df_oficial)}")

Dataset filtrado con éxito. Total de partidos competitivos: 21530


DEFINIR EL TARGET Y TRADUCIR LOS EQUIPOS A NUMEROS

In [93]:
import numpy as np
from sklearn.preprocessing import LabelEncoder  

In [94]:
#Creamos las variables objetivo
#Gana Local = 1, Empate = 0, Gana Visitante = 2
condiciones=[
    (df_oficial["home_score"] > df_oficial["away_score"]),
    (df_oficial["home_score"] == df_oficial["away_score"]),
    (df_oficial["home_score"] < df_oficial["away_score"])   
]
complementos=[1,2,0]
df_oficial["target"]=np.select(condiciones,complementos)

In [95]:
# Inicializamos el codificador para traducir texto de países a números de ID
encoder=LabelEncoder()
equipos=pd.concat([df_oficial["home_team"],df_oficial["away_team"]]).unique()
encoder.fit(equipos)

LabelEncoder()

In [96]:
#Aplicamos la transformación creando columnas numéricas de ID
df_oficial['home_team_code'] = encoder.transform(df_oficial['home_team'])
df_oficial['away_team_code'] = encoder.transform(df_oficial['away_team'])

In [97]:
print("Variable objetivo construida y equipos codificados numéricamente.")

Variable objetivo construida y equipos codificados numéricamente.


In [98]:
# 1. Separamos los datos desde la perspectiva del Local
locales = df_oficial[['match_id', 'date', 'home_team', 'home_score', 'away_score']].copy()
locales.columns = ['match_id', 'date', 'team', 'goles_favor', 'goles_contra']
locales['puntos'] = np.select([locales['goles_favor'] > locales['goles_contra'], locales['goles_favor'] == locales['goles_contra']], [3, 1], default=0)

# 2. Separamos los datos desde la perspectiva del Visitante
visitantes = df_oficial[['match_id', 'date', 'away_team', 'away_score', 'home_score']].copy()
visitantes.columns = ['match_id', 'date', 'team', 'goles_favor', 'goles_contra']
visitantes['puntos'] = np.select([visitantes['goles_favor'] > visitantes['goles_contra'], visitantes['goles_favor'] == visitantes['goles_contra']], [3, 1], default=0)

# 3. Unimos ambas perspectivas en un historial unificado y ordenado cronológicamente por país
historial = pd.concat([locales, visitantes]).sort_values(['team', 'date'])

# 4. Calculamos las métricas móviles de los últimos 5 partidos (restando el partido actual con shift)
historial['racha_5_partidos'] = historial.groupby('team')['puntos'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum()).fillna(0)
historial['promedio_goles_favor'] = historial.groupby('team')['goles_favor'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(0)
historial['promedio_goles_contra'] = historial.groupby('team')['goles_contra'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(0)

# 5. Reincorporamos las métricas al dataset principal mediante cruces de tablas (Merge)
df_oficial = df_oficial.merge(historial[['match_id', 'team', 'racha_5_partidos', 'promedio_goles_favor', 'promedio_goles_contra']], left_on=['match_id', 'home_team'], right_on=['match_id', 'team'], how='left').rename(columns={'racha_5_partidos': 'racha_local', 'promedio_goles_favor': 'goles_favor_local', 'promedio_goles_contra': 'goles_contra_local'}).drop(columns=['team'])
df_oficial = df_oficial.merge(historial[['match_id', 'team', 'racha_5_partidos', 'promedio_goles_favor', 'promedio_goles_contra']], left_on=['match_id', 'away_team'], right_on=['match_id', 'team'], how='left').rename(columns={'racha_5_partidos': 'racha_visitante', 'promedio_goles_favor': 'goles_favor_visitante', 'promedio_goles_contra': 'goles_contra_visitante'}).drop(columns=['team'])

df_ready = df_oficial.dropna().copy()
print("Variables de rendimiento (goles y puntos recientes) consolidadas en el dataset principal.")

Variables de rendimiento (goles y puntos recientes) consolidadas en el dataset principal.


ENTRENAMIENTO DEL MODELO

In [99]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Definimos las columnas que usará el modelo para predecir
features = [
    'home_team_code', 'away_team_code', 
    'racha_local', 'racha_visitante',
    'goles_favor_local', 'goles_contra_local',
    'goles_favor_visitante', 'goles_contra_visitante'
]

X = df_ready[features]
y = df_ready['target']

# 2. Segmentamos los conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 3. Inicializamos y entrenamos el Bosque Aleatorio
modelo_mundial = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_mundial.fit(X_train, y_train)

# 4. Validamos la precisión del modelo entrenado localmente
predicciones = modelo_mundial.predict(X_test)
print(f"¡Modelo entrenado con éxito en tu computadora!")
print(f"Precisión lograda en validación: {accuracy_score(y_test, predicciones) * 100:.2f}%")

¡Modelo entrenado con éxito en tu computadora!
Precisión lograda en validación: 53.35%


PREDECIR PARTIDOS

In [100]:
# 1. Filtramos los partidos de HOY buscando en el calendario completo (df_oficial)
partidos_hoy = df_oficial[df_oficial['date'] == '2026-06-15'].copy()

# 2. Generamos las predicciones con tu modelo ya entrenado
partidos_hoy['pred_codigo'] = modelo_mundial.predict(partidos_hoy[features])

# 3. Traducimos los resultados numéricos a texto
condiciones_hoy = [
    partidos_hoy['pred_codigo'] == 1,
    partidos_hoy['pred_codigo'] == 2,
    partidos_hoy['pred_codigo'] == 0
]
textos_hoy = [
    'Gana ' + partidos_hoy['home_team'],
    'Gana ' + partidos_hoy['away_team'],
    'Empate'
]
partidos_hoy['prediccion_final'] = np.select(condiciones_hoy, textos_hoy)

# 4. Mostramos la cartelera de los partidos de hoy
partidos_hoy[['date', 'home_team', 'away_team', 'prediccion_final']]

,date,home_team,away_team,prediccion_final
21470,2026-06-15,Belgium,Egypt,Gana Belgium
21471,2026-06-15,Iran,New Zealand,Gana Iran
21472,2026-06-15,Spain,Cape Verde,Gana Spain
21473,2026-06-15,Saudi Arabia,Uruguay,Gana Saudi Arabia


INSPECCION VISUAL

In [101]:
# Filtramos la tabla para ver únicamente los partidos del 12 y 13 de junio
fechas_buscadas = ['2026-06-14', '2026-06-15']
comprobacion = df[df['date'].isin(fechas_buscadas)]

# Mostramos las columnas clave
comprobacion[['date', 'home_team', 'away_team', 'home_score', 'away_score']]

C:\Users\ELVA\AppData\Local\Temp\ipykernel_4460\2814235349.py:3: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  comprobacion = df[df['date'].isin(fechas_buscadas)]


,date,home_team,away_team,home_score,away_score
49413,2026-06-14,Germany,Curaçao,7.0,1.0
49414,2026-06-14,Ivory Coast,Ecuador,1.0,0.0
49415,2026-06-14,Netherlands,Japan,2.0,2.0
49416,2026-06-14,Sweden,Tunisia,5.0,1.0
49417,2026-06-15,Belgium,Egypt,NaN,NaN
49418,2026-06-15,Iran,New Zealand,NaN,NaN
49419,2026-06-15,Spain,Cape Verde,NaN,NaN
49420,2026-06-15,Saudi Arabia,Uruguay,NaN,NaN


INTERPRETACION DE RESULTADOS DE MANERA VISUAL

In [102]:
# Filtramos desde el inicio del Mundial (12 de junio) para guardar el historial
partidos_torneo = df_oficial[df_oficial['date'] >= '2026-06-12'].copy()

# Generamos las predicciones
partidos_torneo['pred_codigo'] = modelo_mundial.predict(partidos_torneo[features])

# Traducimos la predicción de la IA a texto
cond_pred = [
    partidos_torneo['pred_codigo'] == 1,
    partidos_torneo['pred_codigo'] == 2,
    partidos_torneo['pred_codigo'] == 0
]
textos_equipos = [
    'Gana ' + partidos_torneo['home_team'],
    'Gana ' + partidos_torneo['away_team'],
    'Empate'
]
partidos_torneo['prediccion_final'] = np.select(cond_pred, textos_equipos)

# Traducimos el resultado REAL a texto (usando tu columna 'target')
cond_real = [
    partidos_torneo['target'] == 1,
    partidos_torneo['target'] == 2,
    partidos_torneo['target'] == 0
]
partidos_torneo['resultado_real'] = np.select(cond_real, textos_equipos)

# Calculamos si la IA acertó o falló comparando ambas columnas
partidos_torneo['estado_prediccion'] = np.where(
    partidos_torneo['prediccion_final'] == partidos_torneo['resultado_real'],
    '✅ Acertó',
    '❌ Falló'
)

# Guardamos la tabla con las nuevas columnas
partidos_torneo.to_csv('predicciones_mundial.csv', index=False)
print("Archivo CSV generado con éxito. ¡Historial y aciertos listos para el Dashboard!")

Archivo CSV generado con éxito. ¡Historial y aciertos listos para el Dashboard!
